In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.schema import HumanMessage
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages


class State(TypedDict):
    files: Annotated[dict, lambda: {}]  # This allows non-message fields
    messages: Annotated[list, list]

# Initialize Gemini model using LangChain
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", convert_system_message_to_human=True)

# Define the chatbot function
def chatbot(state: State) -> State:
    print(state)
    response = llm.invoke(state["messages"])
    return {"messages": [response]}  # `add_messages` will handle appending


def get_state(state: State):
    print(state, state['files'])
# Build the LangGraph
graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("get_state", get_state)
graph_builder.set_entry_point("chatbot")
graph_builder.add_edge('chatbot', 'get_state')
graph = graph_builder.compile()

# Run it with a sample user input
inputs = {"messages": [HumanMessage(content="Hello, what can you do?")]}
for output in graph.stream(inputs):
    print(output)


ValueError: Invalid reducer signature. Expected (a, b) -> c. Got ()

In [ ]:
def stream_graph_updates(user_input: str):
    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}):
        for value in event.values():
            print("Assistant:", value["messages"][-1].content)


while True:
    try:
        user_input = input("User: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_graph_updates(user_input)
    except:
        # fallback if input() is not available
        user_input = "What do you know about LangGraph?"
        print("User: " + user_input)
        stream_graph_updates(user_input)
        break